[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/08_structure/08_structure_lab.ipynb)


# 08 — 구조 검증 — IgFold 로 직접 접고 CDR-H3 RMSD 비교

> 본문 [`08_structure.md`](08_structure.md) 와 **한 절씩 짝지어** 보세요.
> **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 진행합니다.
> 이 노트북의 표·그래프·수치는 **여러분이 방금 돌린 `my_run/`** 에서 나옵니다. 실행을 건너뛰거나 실패하면 저장소에 커밋된 `data/`(실제 실행 산출물) 로 자동 폴백합니다.
>
> **실측 소요 —** IgFold VH+VL 폴딩 **7.1초**(parental) · **12.0초**(humanized)

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`08_structure`) 폴더로 이동한 뒤, 필요한 패키지만 깝니다.
- **로컬**: 이 노트북을 `08_structure/` 폴더 안에서 열었다면 클론 없이 그대로 진행됩니다.

이 셀이 끝나면 두 개의 경로가 준비됩니다 — **`my_run/`**(내가 직접 만들 결과)과 **`data/`**(저장소에 커밋된 레퍼런스 결과).
아래 랩은 항상 `my_run/` 을 먼저 찾고, 없으면 `data/` 로 폴백하면서 **어느 쪽을 쓰는지 print** 합니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "08_structure"
APT_PKGS = ""     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib biopython"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막습니다.
# (멈춤은 예외가 안 나서 폴백이 안 걸립니다 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어집니다)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

# ANARCI 는 numbering 을 hmmscan(HMMER) 서브프로세스로 돌립니다 — pip 만으로는 hmmscan 이 없습니다.
if APT_PKGS and IN_COLAB:
    _run("apt-get -qq update")                 # 인덱스가 낡으면 install 이 404 로 죽는다
    _run(f"apt-get -qq install -y {APT_PKGS}")
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했습니다."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — IgFold 로 parental·humanized 구조 예측

서열 지표가 좋아져도 **CDR loop 모양이 망가지면 결합력이 떨어집니다.** 그래서 만든 서열을 **독립된 모델에게 다시 접어보게** 합니다.

IgFold(+AntiBERTy)로 **VH+VL Fv 하나를 7~12초**에 접습니다(실측: parental 7.1초 · humanized 12.0초).
실행 시 반드시 아래 두 옵션을 끕니다 — 실제로 부딪힌 함정입니다.

| 옵션 | 왜 끄나 |
|---|---|
| `do_refine=False` | `True` 면 **PyRosetta** 를 요구하고, 없으면 그 자리에서 `exit()` 합니다 |
| `do_renum=False` | `True` 면 abnumber 로 재numbering 하는데, 우리 VL 의 C-말단 `G` 가 IMGT 범위 밖이라 **AssertionError** 로 죽습니다 |

스레드도 묶습니다(`OMP_NUM_THREADS=4`) — 과부하 머신에서 IgFold forward 가 간헐적으로 SIGILL 로 죽는 걸 막습니다(실측).

In [ ]:
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["CUDA_VISIBLE_DEVICES"] = ""     # AntiBERTy 가 부모의 try_gpu 를 무시하므로 여기서 차단

_ensure("igfold")
_ensure_pkg_resources()      # IgFold 의존성이 pkg_resources 를 import 합니다(setuptools 81+ 에서 제거됨)

# 후보 서열 — Ch.05 에서 내가 만든 것 우선(가드 없는 argmax; 가드 적용본을 쓰려면 파일명만 바꾸세요)
hp = GUIDE_ROOT / "05_humanize_sapiens" / "my_run" / "sapiens_humanized_noguard.fasta"
if hp.exists():
    print(f"[내 결과 · 05_humanize_sapiens] {hp}")
    f = read_fasta(hp); hum_h, hum_l = f["sapiens_humanized_VH"], f["sapiens_humanized_VL"]
else:
    print("[레퍼런스] data/sapiens_humanized.fasta")
    f = read_fasta("data/sapiens_humanized.fasta")
    hum_h, hum_l = f["sapiens_humanized_VH"], f["sapiens_humanized_VL"]

targets = {"parental": {"H": VH, "L": VL},
           "sapiens_humanized": {"H": hum_h, "L": hum_l}}

try:
    from igfold import IgFoldRunner
    runner = IgFoldRunner()
    for name, seqs in targets.items():
        t0 = time.time()
        runner.fold(str(MY / f"{name}.pdb"), sequences=seqs, do_refine=False, do_renum=False)
        print(f"{name}: {time.time()-t0:.1f}초 → {MY/(name+'.pdb')}")
except Exception as e:
    print("IgFold 실행 실패:", type(e).__name__, str(e)[:200])
    print("→ 아래 분석은 레퍼런스 구조(data/parental.pdb · data/sapiens_humanized.pdb)로 이어집니다.")

## 2) 내 결과 확인 — per-residue 신뢰도 (B-factor = 예측 RMSD)

IgFold 는 PDB 의 B-factor 자리에 **per-residue 예측 오차(Å)** 를 적습니다 — **낮을수록 확신**입니다.
CDR 구간(특히 CDR-H3)에서 값이 튀는 게 정상입니다.

In [ ]:
from Bio.PDB import PDBParser
import pandas as pd, matplotlib.pyplot as plt

parser = PDBParser(QUIET=True)

def prmsd_table(pdb, name):
    st = parser.get_structure(name, str(pdb))[0]
    rows = []
    for ch in st:
        for res in ch:
            if "CA" in res:
                rows.append({"chain": ch.id, "resnum": res.id[1], "resname": res.get_resname(),
                             "prmsd": float(res["CA"].get_bfactor())})
    return pd.DataFrame(rows)

pdbs = {n: find_one(f"{n}.pdb", quiet=False) for n in ("parental", "sapiens_humanized")}
tabs = {n: prmsd_table(p, n) for n, p in pdbs.items()}
for n, t in tabs.items():
    t.to_csv(MY / f"{n}_prmsd.csv", index=False)

ct = pd.read_csv("data/cdr_table_imgt.csv")
h_cdr = [(VH.find(r["sequence"]) + 1, VH.find(r["sequence"]) + len(r["sequence"]))
         for _, r in ct[ct.chain == "H"].iterrows()]

fig, ax = plt.subplots(figsize=(12, 4))
for n, t in tabs.items():
    h = t[t.chain == "H"]
    ax.plot(h["resnum"], h["prmsd"], lw=1.6, label=n)
for lo, hi in h_cdr:
    ax.axvspan(lo, hi, color="#c0508a", alpha=0.12)
ax.set_xlabel("VH residue (raw 1-based)"); ax.set_ylabel("predicted RMSD (Å)  ↓ 좋음")
ax.set_title("IgFold per-residue confidence — VH (분홍 = CDR)", fontweight="bold")
ax.grid(alpha=0.25); ax.legend()
fig.tight_layout(); fig.savefig("08_prmsd.png", dpi=150)
print("CDR 구간(분홍)에서 예측 오차가 커지는 게 정상입니다 — loop 라서.")
for n, t in tabs.items():
    h3 = t[(t.chain == "H") & (t.resnum.between(*h_cdr[2]))]
    print(f"  {n:18s} VH 평균 {t[t.chain=='H'].prmsd.mean():.2f} Å · CDR-H3 평균 {h3.prmsd.mean():.2f} Å")

## 3) 직접 실행 — CDR-H3 backbone RMSD (본문 8.2, 가장 중요한 단일 지표)

**framework CA 로 먼저 정렬**한 뒤, 그 정렬 상태에서 **CDR-H3 CA RMSD** 를 잽니다.
(전체를 한꺼번에 정렬하면 loop 의 변화가 framework 오차에 묻힙니다.)

In [ ]:
from Bio.PDB import Superimposer
import numpy as np

cdr_res = set()
for lo, hi in h_cdr:
    cdr_res |= set(range(lo, hi + 1))
h3_lo, h3_hi = h_cdr[2]

Hp = [r for r in parser.get_structure("p", str(pdbs["parental"]))[0]["H"]]
Hh = [r for r in parser.get_structure("h", str(pdbs["sapiens_humanized"]))[0]["H"]]

fw_p = [r["CA"] for r in Hp if r.id[1] not in cdr_res]
fw_h = [r["CA"] for r in Hh if r.id[1] not in cdr_res]
sup = Superimposer(); sup.set_atoms(fw_p, fw_h)
fw_rmsd = sup.rms
sup.apply([a for r in Hh for a in r])                       # humanized 를 framework 기준으로 이동

h3_p = [r["CA"] for r in Hp if h3_lo <= r.id[1] <= h3_hi]
h3_h = [r["CA"] for r in Hh if h3_lo <= r.id[1] <= h3_hi]
d = np.array([a.coord - b.coord for a, b in zip(h3_p, h3_h)])
h3_rmsd = float(np.sqrt((d ** 2).sum() / len(d)))

allp = [r["CA"] for r in Hp]; allh = [r["CA"] for r in Hh]
sup2 = Superimposer(); sup2.set_atoms(allp, allh)

res = pd.DataFrame([
    {"metric": "framework_fit_rmsd", "value_angstrom": round(fw_rmsd, 4), "n_atoms": len(fw_p)},
    {"metric": "cdr_h3_rmsd_after_framework_alignment", "value_angstrom": round(h3_rmsd, 4), "n_atoms": len(h3_p)},
    {"metric": "whole_vh_rmsd_ca_aligned", "value_angstrom": round(sup2.rms, 4), "n_atoms": len(allp)},
])
res.to_csv(MY / "cdr_h3_rmsd_summary.csv", index=False)
display(res)
print("CDR-H3 RMSD 가 framework RMSD 보다 크면 → humanization 이 loop 를 흔들었다는 뜻입니다.")

## 4) 레퍼런스 대조

In [ ]:
ref = pd.read_csv("data/cdr_h3_rmsd_summary.csv")
print("[레퍼런스 — 실제 IgFold 실행 산출물]")
display(ref)
print("실측: framework 0.2707 Å (91 CA) · CDR-H3 0.5406 Å (13 CA) · VH 전체 0.3207 Å (120 CA)")
print("\n해석 — CDR-H3 가 framework 보다 2배 크게 움직였지만 0.54 Å 수준이면 canonical 구조는 유지된 것으로 봅니다.")
print("       Sapiens 가 CDR-H3 안에 mutation 을 넣지 않았는데도 loop 가 조금 움직인 건, "
      "framework 치환이 loop 받침대(Vernier)를 통해 간접적으로 영향을 주기 때문입니다.")

## 이 랩에서 확인한 것

1. IgFold 로 **VH+VL Fv 를 7~12초**에 접었습니다 — `do_refine=False`(PyRosetta 없음) · `do_renum=False`(VL C-말단 잔기가 IMGT 범위 밖) 가 필수.
2. per-residue 신뢰도(B-factor)는 **CDR 에서 커집니다** — loop 라서 정상.
3. **framework 로 정렬한 뒤 잰 CDR-H3 RMSD = 0.5406 Å**(framework 자체는 0.2707 Å). CDR-H3 에 mutation 이 없는데도 loop 가 움직인 것은 framework 치환의 간접 효과입니다.
4. 이 값이 랭킹(Ch.10)의 **구조 보존 항목**으로 들어갑니다.

다음 → **[Ch.09 — Developability](../09_developability/09_developability_lab.ipynb)**